In [ ]:
# 1
import requests
import json

def get_related_keywords(keyword):
    url = "https://ac.search.naver.com/nx/ac"
    
    params = {
        "q": keyword,
        "_callback": "_jsonp_7", 
        "q_enc": "UTF-8",
        "st": "100"
    }
    
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    
    try:
        res = requests.get(url, params=params, headers=headers)
        res.raise_for_status()

        cleaned_json_str = res.text.split('(', 1)[-1][:-1]
        data = json.loads(cleaned_json_str)
        
        keywords = []
        if data.get('items') and len(data['items']) > 0:
            for item in data['items'][0]:
                keywords.append(item[0].strip()) 
                
        return keywords
        
    except Exception as e:
        return []

if __name__ == "__main__":
    result = get_related_keywords('부트캠프')
    print(result)

['부트캠프', '부트캠프 뜻', '부트캠프 취업', 'ai 부트캠프', '직무부트캠프', '코햄 부트캠프', '마케팅 부트캠프', '맥북 부트캠프', '코멘토 부트캠프', '넷플릭스 부트캠프']


In [ ]:
# 2
import requests
import pandas as pd
import time

URL = 'https://comic.naver.com/api/webtoon/titlelist/weekday'

def fetch(day: str) -> dict:
    res = requests.get(URL, params={'week': day}, headers={
        'User-Agent': 'Mozilla/5.0'
    })
    res.raise_for_status()

    return res.json()

def parse(data: dict, kor_day: str) -> list[dict]:
    webtoons = []

    title_list = data.get('titleList', [])
    
    for item in title_list:
        title_id = item.get('titleId', '')
        
        webtoons.append({
            '제목': item.get('titleName', ''),
            '링크': f"https://comic.naver.com/webtoon/list?titleId={title_id}",
            '요일': kor_day
        })
        
    return webtoons

if __name__ == "__main__":
    week_dict = {
        'mon': '월', 'tue': '화', 'wed': '수',
        'thu': '목', 'fri': '금', 'sat': '토', 'sun': '일', 'dailyPlus': '매일+'
    }
    
    all_webtoons = []
    
    for eng_day, kor_day in week_dict.items():
        data = fetch(eng_day)
        result = parse(data, kor_day)
        
        all_webtoons.extend(result)
        
        time.sleep(0.5)
        
    print(f"총 {len(all_webtoons)}개의 웹툰 수집 완료")
    
    pd.DataFrame(all_webtoons).to_csv(
        "naver_webtoon.csv", 
        index=False, 
        encoding="utf-8-sig"
    )


총 1051개의 웹툰 수집 완료


In [10]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

URL = 'https://www.saramin.co.kr/zf_user/jobs/public/list'

def fetch(page: int) -> dict:
    params = {
        'page': page,
        'isAjaxRequest': 'y'
    }
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'X-Requested-With': 'XMLHttpRequest'
    }
    res = requests.get(URL, params=params, headers=headers)
    res.raise_for_status()
    
    return res.json()

def parse(data: dict) -> list[dict]:
    jobs = []
    
    html_str = data.get('innerHTML', '')
    if not html_str:
        return jobs
    
    soup = BeautifulSoup(html_str, 'html.parser')
    
    items = soup.select('.list_item') 
    
    for item in items:
        corp_tag = item.select_one('.str_tit')             
        group_tag = item.select_one('.main_corp')          
        type_tag = item.select_one('.info_stock')          
        title_tag = item.select_one('.job_tit .str_tit')   
        edu_tag = item.select_one('p.education')           
        exp_tag = item.select_one('p.career')              
        loc_tag = item.select_one('p.work_place')          
        
        corp_name = corp_tag.text.strip() if corp_tag else ''
        group_name = group_tag.text.strip() if group_tag else ''
        corp_type = type_tag.text.strip() if type_tag else ''
        title = title_tag.text.strip() if title_tag else ''
        
        education = edu_tag.text.strip() if edu_tag else ''
        experience = exp_tag.text.strip() if exp_tag else ''
        location = loc_tag.text.strip() if loc_tag else ''
        
        keyword_tags = item.select('.job_sector span')
        keyword_list = []
        for tag in keyword_tags:
            text = tag.text.strip()
            if text:
                keyword_list.append(text)
        
        jobs.append({
            '기업명': corp_name,
            '그룹사': group_name,
            '기업종류': corp_type,
            '공고명': title,
            '직무키워드': keyword_list,
            '학력': education,      
            '경력구분': experience, 
            '근무지': location     
        })
        
    return jobs


if __name__ == "__main__":
    all_jobs = []
    
    for page in range(1, 11):
        try:
            data = fetch(page)
            result = parse(data)
            
            if not result:
                print(f"{page}페이지가 비어 있어 중단합니다.")
                break
                
            all_jobs.extend(result)
            print(f"{page}페이지 수집 완료 ")
            
        except Exception as e:
            print(f"{page}페이지 수집 실패: {e}")
            
        time.sleep(0.5)
        
    print(f"\n총 {len(all_jobs)}건의 사람인 채용공고 수집 완료")
    
    pd.DataFrame(all_jobs).to_csv(
        "saramin.csv", 
        index=False, 
        encoding="utf-8-sig"
    )

1페이지 수집 완료 
2페이지 수집 완료 
3페이지 수집 완료 
4페이지 수집 완료 
5페이지 수집 완료 
6페이지 수집 완료 
7페이지 수집 완료 
8페이지 수집 완료 
9페이지 수집 완료 
10페이지 수집 완료 

총 200건의 사람인 채용공고 수집 완료
